# Bu Dersi Google Colab'da Çalıştır

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BILSEM-BT/Python/blob/main/24-PythonModelDegerlendirme.ipynb)

Bu notebook GitHub üzerinde ders dokümanı olarak yayımlanır. Kodları çalıştırmak ve üzerinde denemeler yapmak için yukarıdaki **Open in Colab** butonunu kullanabilirsiniz.

### Nasıl çalışacağız?

1. **Open in Colab** butonuna tıklayın.
2. Açılan notebook'taki kod hücrelerini `▶` düğmesiyle çalıştırın.
3. Kodları değiştirerek farklı sonuçları deneyin.
4. Çalışmalarınız kendi Colab çalışma alanınızda tutulur; bu GitHub'daki ana ders dosyasını değiştirmez.

> **Önemli:** GitHub'daki bu dosya dersin ana ve değiştirilmeyen kaynağıdır. Colab'da yaptığınız değişiklikler bu dosyaya otomatik olarak yazılmaz.

---

# 24 - Python'da Model Değerlendirme, Cross Validation ve Hiperparametre Optimizasyonu

**Niyazi Sayın BİLSEM**  
**Bilişim Teknolojileri Dersi**  
**Ders Öğretmeni: Ersin ŞANLI**

Önceki derslerde farklı makine öğrenmesi modellerini tek bir train-test ayrımı üzerinden karşılaştırdık.

Bu derste model değerlendirmeyi daha güvenilir hale getireceğiz.

Ana konularımız:

- validation kavramı,
- K-Fold Cross Validation,
- Stratified K-Fold,
- Repeated Stratified K-Fold,
- `cross_val_score()`,
- `cross_validate()`,
- birden fazla metrikle değerlendirme,
- train ve validation skorlarını karşılaştırma,
- `cross_val_predict()`,
- GridSearchCV,
- RandomizedSearchCV,
- Pipeline içinde hiperparametre ayarlama,
- validation curve,
- learning curve,
- final test setinin doğru kullanımı,
- en iyi modeli kaydetme.

Bu dersin sonunda öğrencinin bir modeli yalnızca tek bir test bölmesine bakarak değil, daha sistematik bir değerlendirme süreciyle seçebilmesi hedeflenmektedir.


# 1. Model Değerlendirme Neden Önemlidir?

Bir makine öğrenmesi modelinin eğitim verisinde iyi sonuç vermesi tek başına yeterli değildir.

Asıl amaç:

```text
Daha Önce Görülmemiş Veri
↓
Güvenilir Tahmin
```

üretebilmektir.

Bu nedenle model geliştirme sürecinde eğitim başarısı ile genelleme başarısını birbirinden ayırmalıyız.


# 2. Tek Train-Test Ayrımının Sınırı

Şöyle bir işlem yaptığımızı düşünelim:

```text
Tüm Veri
↓
%80 Train
%20 Test
```

Bu yararlı bir başlangıçtır.

Ancak hangi örneklerin train ve test grubuna düştüğüne göre sonuç bir miktar değişebilir.

Bir model belirli bir ayrımda çok iyi, başka bir ayrımda daha düşük sonuç verebilir.

Cross Validation bu belirsizliği azaltmak için veriyi birden fazla farklı bölümde değerlendirir.


# 3. Train, Validation ve Test Kavramları

Model geliştirmeyi üç farklı amaçla düşünebiliriz.

### Train

Model parametrelerinin öğrenildiği veri.

### Validation

Model ve hiperparametre seçiminde kullanılan veri.

### Test

Model geliştirme tamamlandıktan sonra en son performansı ölçmek için ayrılan veri.

Temel düşünce:

```text
Tüm Veri
│
├── Geliştirme Verisi
│   ├── Train
│   └── Validation
│
└── Final Test
```

Cross Validation, geliştirme verisi içindeki train-validation dönüşümünü sistematik olarak yapabilir.


# 4. Final Test Seti Neden Ayrı Kalmalıdır?

Hiperparametreleri sürekli test setindeki sonuca göre değiştirirsek test seti artık gerçekten görülmemiş veri olmaktan çıkar.

Bu nedenle bu derste:

1. önce final test setini ayıracağız,
2. model seçimini yalnızca eğitim/geliştirme verisinde Cross Validation ile yapacağız,
3. en iyi modeli seçtikten sonra final test setine yalnızca en sonda bakacağız.

Bu yaklaşım yapay zeka projelerinde önemli bir çalışma alışkanlığıdır.


# 5. Gerekli Kütüphaneler

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


# 6. Sınıflandırma Veri Kümesi Oluşturalım

Dersin internet bağlantısından bağımsız çalışması için kontrollü sentetik veri kullanacağız.


In [ ]:
X_array, y_array = make_classification(
    n_samples=800,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_classes=2,
    weights=[0.68, 0.32],
    class_sep=1.1,
    flip_y=0.035,
    random_state=42
)

ozellikler = [
    f"Ozellik{i}"
    for i in range(1, 11)
]

X = pd.DataFrame(
    X_array,
    columns=ozellikler
)

y = pd.Series(
    y_array,
    name="Hedef"
)

X.head()


# 7. Veri Boyutu

In [ ]:
print("X:", X.shape)
print("y:", y.shape)


# 8. Sınıf Dağılımı

In [ ]:
print(
    y.value_counts().sort_index()
)


# 9. Sınıf Oranları

In [ ]:
print(
    y.value_counts(
        normalize=True
    ).sort_index() * 100
)


Sınıflar tamamen dengeli olmadığı için Cross Validation sırasında sınıf oranlarını mümkün olduğunca korumak faydalı olacaktır.


# 10. Önce Final Test Setini Ayırmak

Verinin %20'sini final test seti olarak ayıralım.

Bu veri hiperparametre seçimi sırasında kullanılmayacaktır.


In [ ]:
X_gelistirme, X_test, y_gelistirme, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(
    "Geliştirme:",
    X_gelistirme.shape
)

print(
    "Final test:",
    X_test.shape
)


# 11. Geliştirme Verisinin Sınıf Oranı

In [ ]:
print(
    y_gelistirme.value_counts(
        normalize=True
    ).sort_index()
)


# 12. Final Test Setinin Sınıf Oranı

In [ ]:
print(
    y_test.value_counts(
        normalize=True
    ).sort_index()
)


# 13. K-Fold Cross Validation Nedir?

K-Fold Cross Validation veriyi K parçaya ayırır.

Örneğin K=5:

```text
Fold 1 → Validation
Fold 2-5 → Train

Fold 2 → Validation
Diğerleri → Train

Fold 3 → Validation
Diğerleri → Train

Fold 4 → Validation
Diğerleri → Train

Fold 5 → Validation
Diğerleri → Train
```

Böylece her bölüm bir kez validation verisi olur.


# 14. K-Fold Sonunda Ne Elde Ederiz?

Tek bir skor yerine:

```text
Fold 1 → 0.89
Fold 2 → 0.92
Fold 3 → 0.87
Fold 4 → 0.91
Fold 5 → 0.90
```

gibi birden fazla skor elde ederiz.

Ardından:

- ortalama,
- standart sapma

hesaplayarak modelin genel davranışını inceleyebiliriz.


# 15. KFold ve StratifiedKFold Farkı

`KFold` genel amaçlı K parçalı bölme yapar.

Sınıflandırmada sınıf oranlarını fold'lar arasında korumaya çalışan `StratifiedKFold` çoğunlukla daha uygun bir seçimdir.

Özellikle sınıfların dengeli olmadığı veri kümelerinde stratification önem kazanır.


In [ ]:
from sklearn.model_selection import (
    KFold,
    StratifiedKFold
)


# 16. StratifiedKFold Oluşturmak

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(cv)


Burada:

- `n_splits=5` → 5 fold,
- `shuffle=True` → bölmeden önce örnekleri karıştır,
- `random_state=42` → tekrarlanabilir sonuç

kullanıyoruz.


# 17. Fold'ları Elle İncelemek

Cross Validation bölmelerinin boyutlarını görelim.


In [ ]:
for fold_no, (
    train_index,
    validation_index
) in enumerate(
    cv.split(
        X_gelistirme,
        y_gelistirme
    ),
    start=1
):
    print(
        f"Fold {fold_no}:",
        "Train =",
        len(train_index),
        "Validation =",
        len(validation_index)
    )


# 18. Her Fold'daki Sınıf Oranını Kontrol Etmek

In [ ]:
for fold_no, (
    train_index,
    validation_index
) in enumerate(
    cv.split(
        X_gelistirme,
        y_gelistirme
    ),
    start=1
):
    validation_y = y_gelistirme.iloc[
        validation_index
    ]

    oranlar = validation_y.value_counts(
        normalize=True
    ).sort_index()

    print(
        f"Fold {fold_no}:",
        oranlar.to_dict()
    )


StratifiedKFold sınıf dağılımını fold'larda yaklaşık olarak korumaya çalışır.


# 19. İlk Model: Logistic Regression Pipeline

Ölçeklendirme ile modeli Pipeline içinde birleştirelim.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lojistik = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

print(lojistik)


# 20. `cross_val_score()`

Bir modeli Cross Validation ile tek bir metrik üzerinden değerlendirmek için `cross_val_score()` kullanabiliriz.


In [ ]:
from sklearn.model_selection import cross_val_score

f1_skorlari = cross_val_score(
    lojistik,
    X_gelistirme,
    y_gelistirme,
    cv=cv,
    scoring="f1"
)

print(f1_skorlari)


# 21. Fold Sonuçlarını DataFrame'e Aktarmak

In [ ]:
f1_df = pd.DataFrame({
    "Fold": range(
        1,
        len(f1_skorlari) + 1
    ),
    "F1": f1_skorlari
})

f1_df


# 22. Ortalama Cross Validation Skoru

In [ ]:
print(
    "Ortalama F1:",
    f1_skorlari.mean()
)


# 23. Standart Sapma

In [ ]:
print(
    "F1 standart sapma:",
    f1_skorlari.std()
)


Ortalama başarı modelin genel seviyesini gösterirken fold'lar arasındaki değişim de modelin değerlendirme kararlılığı hakkında bilgi verebilir.


# 24. Cross Validation Sonuçlarını Grafikleştirmek

In [ ]:
plt.bar(
    f1_df["Fold"].astype(str),
    f1_df["F1"]
)

plt.ylim(0, 1)
plt.xlabel("Fold")
plt.ylabel("F1")
plt.title("Cross Validation Fold Sonuçları")
plt.show()


# 25. Accuracy ile Cross Validation

In [ ]:
accuracy_skorlari = cross_val_score(
    lojistik,
    X_gelistirme,
    y_gelistirme,
    cv=cv,
    scoring="accuracy"
)

print(
    accuracy_skorlari
)

print(
    "Ortalama Accuracy:",
    accuracy_skorlari.mean()
)


# 26. `cross_validate()`

`cross_val_score()` tek bir skor için kullanışlıdır.

Birden fazla metriği aynı anda değerlendirmek istediğimizde `cross_validate()` kullanabiliriz.


In [ ]:
from sklearn.model_selection import cross_validate


# 27. Birden Fazla Metrik Belirlemek

In [ ]:
skorlar = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

cv_sonuclari = cross_validate(
    lojistik,
    X_gelistirme,
    y_gelistirme,
    cv=cv,
    scoring=skorlar,
    return_train_score=True
)

print(
    cv_sonuclari.keys()
)


Sonuç sözlüğünde:

- fit süresi,
- score süresi,
- train skorları,
- validation/test skorları

bulunabilir.


# 28. Cross Validate Sonuçlarını DataFrame'e Çevirmek

In [ ]:
cv_df = pd.DataFrame(
    cv_sonuclari
)

cv_df


# 29. Ortalama Metrikler

In [ ]:
ortalama_metrikler = pd.DataFrame({
    "Metrik": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ],
    "Train": [
        cv_df["train_accuracy"].mean(),
        cv_df["train_precision"].mean(),
        cv_df["train_recall"].mean(),
        cv_df["train_f1"].mean()
    ],
    "Validation": [
        cv_df["test_accuracy"].mean(),
        cv_df["test_precision"].mean(),
        cv_df["test_recall"].mean(),
        cv_df["test_f1"].mean()
    ]
})

ortalama_metrikler


# 30. Train ve Validation Skorlarını Karşılaştırmak

Eğitim skoru validation skorundan çok daha yüksekse overfitting ihtimali araştırılabilir.


In [ ]:
ortalama_metrikler.set_index(
    "Metrik"
).plot(
    kind="bar",
    figsize=(9, 5)
)

plt.ylim(0, 1)
plt.ylabel("Skor")
plt.title("Train ve Validation Skorları")
plt.xticks(rotation=0)
plt.show()


# 31. Repeated Stratified K-Fold

Tek bir 5-fold bölme yerine Cross Validation işlemini farklı karıştırmalarla birden fazla kez tekrar edebiliriz.


In [ ]:
from sklearn.model_selection import (
    RepeatedStratifiedKFold
)

tekrarli_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

tekrarli_f1 = cross_val_score(
    lojistik,
    X_gelistirme,
    y_gelistirme,
    cv=tekrarli_cv,
    scoring="f1"
)

print(
    "Toplam skor sayısı:",
    len(tekrarli_f1)
)

print(
    "Ortalama F1:",
    tekrarli_f1.mean()
)

print(
    "Standart sapma:",
    tekrarli_f1.std()
)


5 fold × 3 tekrar sonucunda 15 değerlendirme skoru elde ettik.


# 32. Tek Bölme ile Cross Validation Sonucunu Karşılaştırmak

Önce basit bir train-validation ayrımı yapalım.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_gelistirme,
    y_gelistirme,
    test_size=0.20,
    random_state=42,
    stratify=y_gelistirme
)

tek_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

tek_model.fit(
    X_train,
    y_train
)

print(
    "Tek validation F1:",
    __import__(
        "sklearn.metrics",
        fromlist=["f1_score"]
    ).f1_score(
        y_val,
        tek_model.predict(X_val)
    )
)

print(
    "5-Fold Ortalama F1:",
    f1_skorlari.mean()
)


Tek validation skoru yalnızca bir bölmenin sonucudur.

Cross Validation birden fazla bölmede değerlendirme yaptığı için daha zengin bilgi sağlar.


# 33. `cross_val_predict()`

Her örnek için, o örneğin eğitimde bulunmadığı fold'daki modelin tahminini üretmek mümkündür.

Buna out-of-fold tahmini gibi düşünebiliriz.


In [ ]:
from sklearn.model_selection import (
    cross_val_predict
)

oof_tahmin = cross_val_predict(
    lojistik,
    X_gelistirme,
    y_gelistirme,
    cv=cv,
    method="predict"
)

print(
    oof_tahmin[:20]
)


# 34. Out-of-Fold Confusion Matrix

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)

oof_cm = confusion_matrix(
    y_gelistirme,
    oof_tahmin
)

print(oof_cm)


In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=oof_cm,
    display_labels=[
        "Sınıf 0",
        "Sınıf 1"
    ]
).plot()

plt.title(
    "Cross Validation Out-of-Fold Tahminleri"
)
plt.show()


# 35. Hiperparametre Nedir?

Modelin eğitim sırasında veriden doğrudan öğrenmediği, bizim belirlediğimiz ayarlara **hiperparametre** denir.

Örnekler:

### Logistic Regression

```text
C
class_weight
```

### KNN

```text
n_neighbors
weights
```

### Decision Tree

```text
max_depth
min_samples_split
min_samples_leaf
```

### Random Forest

```text
n_estimators
max_depth
max_features
```


# 36. Parametre ve Hiperparametre Farkı

### Model Parametreleri

`fit()` sırasında veriden öğrenilir.

Örneğin Logistic Regression katsayıları.

### Hiperparametreler

Model eğitilmeden önce belirlenir.

Örneğin:

```python
LogisticRegression(C=1.0)
```

Buradaki `C` bir hiperparametredir.


# 37. Hiperparametreyi Test Setine Bakarak Seçmeyelim

Yanlış süreç:

```text
Model kur
↓
Test et
↓
Hiperparametre değiştir
↓
Aynı test setine tekrar bak
↓
Tekrar değiştir
```

Böyle yaptıkça test verisini model geliştirme sürecine dahil etmiş oluruz.

Doğru yaklaşım:

```text
Geliştirme verisi
↓
Cross Validation
↓
Hiperparametre seçimi
↓
En iyi model
↓
Final test
```


# 38. Grid Search Nedir?

Grid Search, bizim verdiğimiz hiperparametre değerlerinin bütün kombinasyonlarını sistematik olarak dener.

Örnek:

```text
C = [0.01, 0.1, 1, 10]
class_weight = [None, balanced]
```

Toplam:

```text
4 × 2 = 8
```

kombinasyon denenir.

Her kombinasyon Cross Validation ile değerlendirilir.


# 39. Pipeline Parametre İsimleri

Pipeline içindeki adımlarımız:

```python
("scaler", StandardScaler())
("model", LogisticRegression())
```

Bu nedenle Logistic Regression'ın `C` parametresine GridSearch içinde:

```text
model__C
```

şeklinde erişiriz.

İki alt çizgi:

```text
__
```

Pipeline adımı ile parametre adını ayırır.


# 40. GridSearchCV'yi İçe Aktarmak

In [ ]:
from sklearn.model_selection import GridSearchCV


# 41. Grid Search Parametre Tablosu

In [ ]:
parametre_grid = {
    "model__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ],
    "model__class_weight": [
        None,
        "balanced"
    ]
}

parametre_grid


# 42. GridSearchCV Oluşturmak

In [ ]:
grid_search = GridSearchCV(
    estimator=lojistik,
    param_grid=parametre_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

print(grid_search)


# 43. Grid Search'ü Eğitmek

Dikkat: yalnızca geliştirme verisini kullanıyoruz.

Final test setine dokunmuyoruz.


In [ ]:
grid_search.fit(
    X_gelistirme,
    y_gelistirme
)

print("Grid Search tamamlandı.")


# 44. En İyi Parametreler

In [ ]:
print(
    "En iyi parametreler:"
)

print(
    grid_search.best_params_
)


# 45. En İyi Cross Validation Skoru

In [ ]:
print(
    "En iyi CV F1:",
    grid_search.best_score_
)


`best_score_` final test skoru değildir.

Grid Search sırasında Cross Validation'dan elde edilen en iyi ortalama skordur.


# 46. En İyi Model

In [ ]:
en_iyi_lojistik = (
    grid_search.best_estimator_
)

print(en_iyi_lojistik)


GridSearchCV varsayılan `refit=True` davranışıyla seçilen en iyi parametrelerle modeli geliştirme verisi üzerinde yeniden eğitebilir.

Bu `best_estimator_` nesnesini doğrudan tahmin için kullanabiliriz.


# 47. Bütün Grid Search Sonuçları

In [ ]:
grid_sonuclari = pd.DataFrame(
    grid_search.cv_results_
)

print(
    grid_sonuclari.columns.tolist()
)


# 48. Önemli Sütunları Görmek

In [ ]:
grid_ozet = grid_sonuclari[
    [
        "param_model__C",
        "param_model__class_weight",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
)

grid_ozet


# 49. Grid Search Sonuç Grafiği

In [ ]:
grafik_veri = grid_ozet.copy()

grafik_veri["Ayar"] = (
    grafik_veri[
        "param_model__C"
    ].astype(str)
    +
    " / "
    +
    grafik_veri[
        "param_model__class_weight"
    ].astype(str)
)

plt.figure(figsize=(11, 5))

plt.bar(
    grafik_veri["Ayar"],
    grafik_veri["mean_test_score"]
)

plt.ylabel("Ortalama CV F1")
plt.xlabel("C / class_weight")
plt.title("Grid Search Sonuçları")
plt.xticks(rotation=45)
plt.show()


# 50. Grid Search Maliyeti

Grid Search bütün kombinasyonları dener.

Örneğin:

```text
20 parametre kombinasyonu
×
5 fold
=
100 model eğitimi
```

Parametre sayısı büyüdükçe hesaplama maliyeti hızla artabilir.

Bu nedenle daha büyük arama alanlarında RandomizedSearchCV yararlı olabilir.


# 51. Randomized Search Nedir?

Randomized Search bütün kombinasyonları denemek yerine belirlenen sayıda parametre kombinasyonunu rastgele örnekler.

Örneğin:

```text
1000 olası kombinasyon
↓
n_iter=20
↓
20 farklı kombinasyon dene
```

Bu yaklaşım büyük hiperparametre uzaylarında zaman kazandırabilir.


# 52. Random Forest ile Randomized Search

Şimdi Random Forest için daha geniş bir hiperparametre alanı oluşturalım.


In [ ]:
from sklearn.ensemble import (
    RandomForestClassifier
)

orman = RandomForestClassifier(
    random_state=42
)

print(orman)


# 53. Randomized Search Parametreleri

In [ ]:
parametre_dagilimlari = {
    "n_estimators": [
        100,
        150,
        200,
        250,
        300
    ],
    "max_depth": [
        None,
        3,
        5,
        7,
        10,
        15
    ],
    "min_samples_split": [
        2,
        4,
        6,
        10
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ],
    "max_features": [
        "sqrt",
        "log2",
        None
    ],
    "class_weight": [
        None,
        "balanced"
    ]
}

parametre_dagilimlari


# 54. RandomizedSearchCV

In [ ]:
from sklearn.model_selection import (
    RandomizedSearchCV
)

random_search = RandomizedSearchCV(
    estimator=orman,
    param_distributions=parametre_dagilimlari,
    n_iter=15,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)

print(random_search)


# 55. Randomized Search'ü Çalıştırmak

In [ ]:
random_search.fit(
    X_gelistirme,
    y_gelistirme
)

print(
    "Randomized Search tamamlandı."
)


# 56. En İyi Random Forest Parametreleri

In [ ]:
print(
    random_search.best_params_
)


# 57. En İyi Random Forest CV Skoru

In [ ]:
print(
    "En iyi CV F1:",
    random_search.best_score_
)


# 58. En İyi Random Forest Modeli

In [ ]:
en_iyi_orman = (
    random_search.best_estimator_
)

print(en_iyi_orman)


# 59. Randomized Search Sonuçlarını İncelemek

In [ ]:
random_sonuclari = pd.DataFrame(
    random_search.cv_results_
)

random_ozet = random_sonuclari[
    [
        "params",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
)

random_ozet.head(10)


# 60. Grid Search ve Randomized Search Farkı

### GridSearchCV

- verilen bütün kombinasyonları dener,
- küçük ve kontrollü arama alanlarında uygundur,
- arama maliyeti kombinasyon sayısıyla büyür.

### RandomizedSearchCV

- belirli sayıda kombinasyonu örnekler,
- `n_iter` ile deneme sayısı kontrol edilir,
- büyük arama alanlarında daha ekonomik olabilir.

Her iki yöntem de Cross Validation ile hiperparametre seçimi yapabilir.


# 61. İki Optimize Modeli Cross Validation Skoruyla Karşılaştırmak

In [ ]:
arama_karsilastirma = pd.DataFrame({
    "Model": [
        "Grid Logistic Regression",
        "Randomized Random Forest"
    ],
    "EnIyiCV_F1": [
        grid_search.best_score_,
        random_search.best_score_
    ]
})

arama_karsilastirma


Bu karşılaştırma final test sonucu değildir.

İki model de geliştirme verisindeki Cross Validation sonucuyla karşılaştırılmaktadır.


# 62. Final Test Aşaması

Artık model seçimi tamamlandı.

Şimdi final test setini ilk kez kullanabiliriz.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


# 63. Grid Search Logistic Regression Final Test

In [ ]:
lojistik_test_pred = (
    en_iyi_lojistik.predict(
        X_test
    )
)

print(
    "Final Test F1:",
    f1_score(
        y_test,
        lojistik_test_pred
    )
)


# 64. Randomized Search Random Forest Final Test

In [ ]:
orman_test_pred = (
    en_iyi_orman.predict(
        X_test
    )
)

print(
    "Final Test F1:",
    f1_score(
        y_test,
        orman_test_pred
    )
)


# 65. Final Test Karşılaştırma Tablosu

In [ ]:
final_sonuclar = pd.DataFrame({
    "Model": [
        "Tuned Logistic Regression",
        "Tuned Random Forest"
    ],
    "Accuracy": [
        accuracy_score(
            y_test,
            lojistik_test_pred
        ),
        accuracy_score(
            y_test,
            orman_test_pred
        )
    ],
    "Precision": [
        precision_score(
            y_test,
            lojistik_test_pred
        ),
        precision_score(
            y_test,
            orman_test_pred
        )
    ],
    "Recall": [
        recall_score(
            y_test,
            lojistik_test_pred
        ),
        recall_score(
            y_test,
            orman_test_pred
        )
    ],
    "F1": [
        f1_score(
            y_test,
            lojistik_test_pred
        ),
        f1_score(
            y_test,
            orman_test_pred
        )
    ]
})

final_sonuclar


# 66. Final Test Classification Report

Örnek olarak Random Forest için:


In [ ]:
print(
    classification_report(
        y_test,
        orman_test_pred,
        target_names=[
            "Sınıf 0",
            "Sınıf 1"
        ]
    )
)


# 67. Final Test Confusion Matrix

In [ ]:
final_cm = confusion_matrix(
    y_test,
    orman_test_pred
)

ConfusionMatrixDisplay(
    confusion_matrix=final_cm,
    display_labels=[
        "Sınıf 0",
        "Sınıf 1"
    ]
).plot()

plt.title(
    "Tuned Random Forest - Final Test"
)
plt.show()


# 68. Cross Validation ile Final Test Arasındaki Fark

Cross Validation skoru:

```text
model seçimi sırasında
geliştirme verisinden
```

elde edilir.

Final test skoru:

```text
model seçimi tamamlandıktan sonra
ayrı tutulmuş test verisinden
```

elde edilir.

Bu iki skorun birbirine yakın olması modelin genelleme davranışı açısından olumlu bir işaret olabilir.


# 69. Validation Curve Nedir?

Bir hiperparametrenin farklı değerlerinde:

- train skoru,
- validation skoru

nasıl değişiyor görmek için validation curve kullanılabilir.

Bu bize:

- underfitting,
- uygun karmaşıklık,
- overfitting

hakkında fikir verebilir.


# 70. Logistic Regression C Değeri İçin Validation Curve

In [ ]:
from sklearn.model_selection import (
    validation_curve
)

c_degerleri = np.logspace(
    -3,
    3,
    7
)

train_scores, val_scores = (
    validation_curve(
        lojistik,
        X_gelistirme,
        y_gelistirme,
        param_name="model__C",
        param_range=c_degerleri,
        cv=cv,
        scoring="f1",
        n_jobs=-1
    )
)

print(
    train_scores.shape,
    val_scores.shape
)


# 71. Validation Curve Ortalamaları

In [ ]:
validation_df = pd.DataFrame({
    "C": c_degerleri,
    "TrainF1": train_scores.mean(
        axis=1
    ),
    "ValidationF1": val_scores.mean(
        axis=1
    )
})

validation_df


# 72. Validation Curve Grafiği

In [ ]:
plt.semilogx(
    validation_df["C"],
    validation_df["TrainF1"],
    marker="o",
    label="Train F1"
)

plt.semilogx(
    validation_df["C"],
    validation_df[
        "ValidationF1"
    ],
    marker="o",
    label="Validation F1"
)

plt.xlabel("C")
plt.ylabel("F1")
plt.title(
    "Logistic Regression Validation Curve"
)
plt.legend()
plt.grid()
plt.show()


# 73. Learning Curve Nedir?

Learning Curve, modelin farklı eğitim veri miktarlarında nasıl davrandığını gösterir.

Sorular:

- Daha fazla veri faydalı olabilir mi?
- Train ve validation skorları birbirine yaklaşıyor mu?
- Model veri azlığından etkileniyor mu?

gibi konularda fikir verebilir.


# 74. Learning Curve Hesaplamak

In [ ]:
from sklearn.model_selection import (
    learning_curve
)

train_sizes, train_scores_lc, val_scores_lc = (
    learning_curve(
        en_iyi_lojistik,
        X_gelistirme,
        y_gelistirme,
        cv=cv,
        scoring="f1",
        train_sizes=np.linspace(
            0.2,
            1.0,
            5
        ),
        n_jobs=-1
    )
)

print(train_sizes)


# 75. Learning Curve DataFrame

In [ ]:
learning_df = pd.DataFrame({
    "TrainSize": train_sizes,
    "TrainF1": train_scores_lc.mean(
        axis=1
    ),
    "ValidationF1": val_scores_lc.mean(
        axis=1
    )
})

learning_df


# 76. Learning Curve Grafiği

In [ ]:
plt.plot(
    learning_df["TrainSize"],
    learning_df["TrainF1"],
    marker="o",
    label="Train F1"
)

plt.plot(
    learning_df["TrainSize"],
    learning_df[
        "ValidationF1"
    ],
    marker="o",
    label="Validation F1"
)

plt.xlabel(
    "Eğitim Örneği Sayısı"
)
plt.ylabel("F1")
plt.title("Learning Curve")
plt.legend()
plt.grid()
plt.show()


# 77. Overfitting'i Skorlardan Okumak

Genel bir işaret:

```text
Train skoru çok yüksek
Validation skoru belirgin düşük
```

ise model eğitim verisine fazla uyum sağlamış olabilir.

Bu tek başına kesin teşhis değildir ancak araştırılması gereken önemli bir işarettir.


# 78. Underfitting'i Skorlardan Okumak

Eğer:

```text
Train skoru düşük
Validation skoru da düşük
```

ise model problem için yeterince güçlü olmayabilir veya kullanılan özellikler yeterli bilgi içermiyor olabilir.

Model karmaşıklığı artırmak tek çözüm değildir.

Veri ve özellik kalitesi de değerlendirilmelidir.


# 79. Skorun Standart Sapması Neden Önemlidir?

İki model düşünelim:

```text
Model A:
Ortalama F1 = 0.90
Std = 0.01

Model B:
Ortalama F1 = 0.91
Std = 0.08
```

Model B ortalamada biraz daha yüksek olsa da fold'lar arasında çok daha değişken olabilir.

Model seçerken yalnızca ortalama skor değil, kararlılık da incelenebilir.


# 80. Çoklu Metrikle Hiperparametre Araması

Gerçek projelerde birden fazla metriği aynı anda hesaplamak mümkündür.

Örneğin:

```python
scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "recall": "recall"
}
```

Bir metriği model seçimi için `refit` metriği olarak belirleyebiliriz.

Bu derste ana seçim metriğimiz F1 olduğu için temel örneklerde `scoring="f1"` kullandık.


# 81. Model Seçim Metriği Probleme Göre Değişir

Her zaman F1 seçmek zorunda değiliz.

Örneğin:

### Yanlış negatif çok maliyetliyse

Recall daha önemli olabilir.

### Yanlış pozitif çok maliyetliyse

Precision daha önemli olabilir.

### Sınıflar dengeli ve hatalar benzer maliyetteyse

Accuracy yararlı olabilir.

Model metriği teknik alışkanlıkla değil, problem hedefiyle seçilmelidir.


# 82. Regresyonda Cross Validation

Cross Validation yalnızca sınıflandırma için değildir.

Regresyon modellerinde de kullanılabilir.

Örneğin:

```python
cross_val_score(
    regresyon_modeli,
    X,
    y,
    cv=5,
    scoring="neg_mean_absolute_error"
)
```

Scikit-learn bazı hata metriklerinde skorların daha büyük değerinin daha iyi olması kuralını korumak için negatif skor isimleri kullanır.

Sonuç yorumlanırken işaretin anlamına dikkat etmek gerekir.


# 83. Zaman Serilerinde Normal K-Fold Kullanılır mı?

Zaman sıralı verilerde gelecekteki verinin geçmiş veriyi tahmin eden modele yanlış biçimde karışmaması gerekir.

Bu nedenle zaman serilerinde rastgele K-Fold yerine zaman sırasını koruyan özel bölme yöntemleri tercih edilir.

Scikit-learn bu amaçla `TimeSeriesSplit` gibi araçlar sağlar.

Zaman serisi tahminini ayrı bir ileri yapay zeka dersinde ele alacağız.


# 84. Gruplu Verilerde Dikkat

Aynı kişiye, cihaza veya gruba ait çok sayıda kayıt varsa aynı grubun train ve validation tarafına dağılması bazen veri sızıntısına yol açabilir.

Bu tür durumlarda grup bilgisini dikkate alan Cross Validation yöntemleri gerekebilir.

Model değerlendirme stratejisi veri üretim biçimine göre seçilmelidir.


# 85. Nested Cross Validation Kavramına İlk Bakış

Daha ileri çalışmalar için:

```text
İç Cross Validation
→ Hiperparametre seçimi

Dış Cross Validation
→ Seçim sürecinin genel performansını değerlendirme
```

şeklinde Nested Cross Validation kullanılabilir.

Bu dersin temel uygulamalarında ayrı final test seti + geliştirme verisinde Cross Validation yaklaşımını kullanıyoruz.


# 86. En İyi Modeli Kaydetmek

Final değerlendirme sonrasında kullanmak istediğimiz modeli dosyaya kaydedebiliriz.


In [ ]:
import joblib

joblib.dump(
    en_iyi_orman,
    "24-en-iyi-siniflandirma-modeli.joblib"
)

print("Model kaydedildi.")


# 87. Modeli Yüklemek

In [ ]:
yuklenen_model = joblib.load(
    "24-en-iyi-siniflandirma-modeli.joblib"
)

print(yuklenen_model)


# 88. Yeni Veri İçin Tahmin

In [ ]:
yeni_ornek = pd.DataFrame(
    [[
        0.2,
        -0.4,
        1.1,
        0.7,
        -0.8,
        0.5,
        1.6,
        -1.0,
        0.3,
        0.9
    ]],
    columns=ozellikler
)

sinif = yuklenen_model.predict(
    yeni_ornek
)[0]

print(
    "Tahmin edilen sınıf:",
    sinif
)


# 89. Tahmin Olasılıkları

In [ ]:
olasilik = (
    yuklenen_model.predict_proba(
        yeni_ornek
    )[0]
)

print(
    "Sınıf 0:",
    round(
        float(olasilik[0]),
        3
    )
)

print(
    "Sınıf 1:",
    round(
        float(olasilik[1]),
        3
    )
)


# 90. Model Geliştirme Akışı

Artık model geliştirme sürecimiz daha düzenli:

```text
Problem
↓
Veri
↓
Final Test Setini Ayır
↓
Geliştirme Verisi
↓
Cross Validation
↓
Baseline
↓
Model Karşılaştırma
↓
Hiperparametre Arama
↓
GridSearchCV / RandomizedSearchCV
↓
Validation Curve
↓
Learning Curve
↓
En İyi Model
↓
Final Test
↓
Modeli Kaydet
↓
Uygulamaya Entegre Et
```


# 91. Sık Yapılan Hatalar

### Test setiyle hiperparametre seçmek

Final testin tarafsızlığını bozar.

### Ölçeklendirmeyi bütün veri üzerinde öğrenmek

Veri sızıntısı oluşturabilir.

### Sadece accuracy kullanmak

Dengesiz sınıflarda yanıltıcı olabilir.

### En yüksek tek fold skorunu seçmek

Cross Validation ortalaması ve değişkenliği birlikte değerlendirilmelidir.

### Çok büyük Grid Search alanı oluşturmak

Gereksiz hesaplama maliyeti oluşturabilir.

### `best_score_` değerini final test skoru sanmak

Bu değer Cross Validation seçim skorudur.

### Pipeline parametre adında `__` kullanımını unutmak

Grid Search doğru parametreye ulaşamaz.


# 92. Grid Search mi Randomized Search mü?

### Grid Search seçilebilir:

- az sayıda hiperparametre varsa,
- denenmesi gereken değerler netse,
- kombinasyon sayısı yönetilebilirse.

### Randomized Search seçilebilir:

- hiperparametre alanı büyükse,
- bütün kombinasyonları denemek pahalıysa,
- belirli bir hesaplama bütçesi varsa.

İki yöntem birbirinin rakibi olmak zorunda değildir.

Örneğin önce Randomized Search ile iyi bölge bulunup ardından daha dar bir Grid Search yapılabilir.


# 93. Ders Özeti

Bu derste:

- train,
- validation,
- test,
- K-Fold,
- StratifiedKFold,
- RepeatedStratifiedKFold,
- `cross_val_score()`,
- `cross_validate()`,
- `cross_val_predict()`,
- out-of-fold tahmin,
- ortalama skor,
- standart sapma,
- train-validation karşılaştırması,
- hiperparametre,
- GridSearchCV,
- Pipeline parametre isimleri,
- `best_params_`,
- `best_score_`,
- `best_estimator_`,
- `cv_results_`,
- RandomizedSearchCV,
- `n_iter`,
- final test seti,
- validation curve,
- learning curve,
- overfitting,
- underfitting,
- model seçme metriği,
- model kaydetme ve yükleme

konularını öğrendik.


# 94. Mini Uygulamalar

1. 1000 örnekli kendi sınıflandırma veri kümenizi oluşturun.
2. Final test setini başta ayırın.
3. 5-fold StratifiedKFold oluşturun.
4. Fold boyutlarını yazdırın.
5. Her fold'un sınıf oranını kontrol edin.
6. Logistic Regression için `cross_val_score()` kullanın.
7. Accuracy için 5-fold skorlarını hesaplayın.
8. F1 için 5-fold skorlarını hesaplayın.
9. Ortalama ve standart sapmayı hesaplayın.
10. `cross_validate()` ile dört metriği aynı anda ölçün.
11. Train ve validation skorlarını karşılaştırın.
12. RepeatedStratifiedKFold ile 15 skor üretin.
13. `cross_val_predict()` ile out-of-fold tahmin üretin.
14. Out-of-fold confusion matrix oluşturun.
15. Logistic Regression için GridSearchCV oluşturun.
16. En iyi `C` değerini bulun.
17. `class_weight` parametresini Grid Search'e ekleyin.
18. `cv_results_` sonuçlarını DataFrame'e dönüştürün.
19. Random Forest için RandomizedSearchCV oluşturun.
20. En az 10 farklı Random Forest ayarı deneyin.
21. Grid Search ve Randomized Search sonuçlarını karşılaştırın.
22. Validation Curve oluşturun.
23. Learning Curve oluşturun.
24. En iyi modeli final test setinde bir kez değerlendirin.
25. En iyi modeli `joblib` ile kaydedip yeniden yükleyin.


# 95. Yapay Zeka Proje Görevi

Bir önceki sınıflandırma projenizi **bilimsel model değerlendirme süreci** ekleyerek geliştirin.

Projede en az:

- 700 örnek,
- en az 5 özellik,
- ayrı final test seti,
- Stratified K-Fold,
- baseline model,
- en az 3 farklı sınıflandırıcı,
- `cross_val_score`,
- `cross_validate`,
- accuracy,
- precision,
- recall,
- F1,
- Cross Validation ortalama ve standart sapması,
- GridSearchCV,
- RandomizedSearchCV,
- en iyi parametreler,
- final test değerlendirmesi,
- confusion matrix,
- classification report,
- model dosyasına kaydetme

bulunsun.

Ek geliştirme:

- validation curve,
- learning curve,
- Flask tahmin arayüzü,
- model değerlendirme sonuçlarını SQLite'a kaydetme

özelliklerinden biri eklenebilir.


# Dersin Ana Kazanımı

Bu dersin sonunda öğrencinin model geliştirme sürecini şu şekilde kurabilmesi hedeflenmektedir:

**Veriyi Ayır**

↓

**Final Testi Koruma Altına Al**

↓

**Cross Validation**

↓

**Birden Fazla Metrik**

↓

**Model Karşılaştırma**

↓

**Hiperparametre Arama**

↓

**GridSearchCV / RandomizedSearchCV**

↓

**Overfitting / Underfitting Analizi**

↓

**Final Test**

↓

**En İyi Modeli Kaydet**

Bu aşamadan sonra artık yalnızca model eğiten değil, modelin gerçekten ne kadar güvenilir olduğunu sistematik biçimde ölçmeye çalışan bir yapay zeka geliştirme sürecine geçmiş oluyoruz.

Bir sonraki derste **denetimsiz öğrenme ve K-Means kümeleme** uygulamalarına geçeceğiz.
